# 전체 PDF 주제 청킹 미리보기

전체 779쪽을 읽고, 목차의 제목과 본문에 인쇄된 쪽수를 대조해 주제 경계를 찾습니다. 여러 페이지에 이어지는 글은 다음 제목까지 한 묶음으로 기록합니다.

앞서 확인한 안내도·표·글 예제는 재사용합니다. 나머지는 **자동으로 만든 검토용 초안**입니다. 제목 누락·표 구조·경고문 경계·그림 연결은 추가 확인이 필요하므로 자동 초안을 최종 답변 자료로 승인하지 않습니다.

그림은 PDF 페이지와 내부 이미지 키로 참조합니다. 모든 이미지를 파일로 저장하거나 설명을 자동 생성하지 않습니다. 청크와 벡터 파일·Supabase 저장도 없습니다.

## 수업 기반과 프로젝트 추가 기록 (2026-10-01)

코드의 추가 설명은 주석으로 남겼습니다. 이 표시는 프로젝트 적용 기록이며, 수업에서 개념 자체를 다루지 않았다는 뜻은 아닙니다.

- 싼타페 목차 제목·인쇄 쪽수·장 번호를 이용해 전체 본문의 자동 주제 초안을 만듭니다.
- 추가: 페이지 용도 구분, 확인 예제 중복 제외, 꼬리말 모호성 표시, 페이지 수준 이미지 참조와 검토 상태.

전체 목록: [수업 기반·추가 적용 기록](../src/car_search_rag/zzong_santafe_lag/learning_additions.md)


### 1. 확인한 예제와 모델 길이 준비

04·05번 노트북의 코드로 검증한 기록과 함수를 준비합니다. PDF 전체 처리는 이 노트북에서 수행하며, 이전 전체 미리보기를 다시 실행하지 않습니다.

In [1]:
# [프로젝트 추가 기록] 싼타페 목차 제목·인쇄 쪽수·장 번호를 이용해 전체 본문의 자동 주제 초안을 만듭니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
from pathlib import Path
import json
import contextlib
import io
import re
from collections import Counter, defaultdict
from functools import lru_cache

project_folder = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

def run_notebook_source(filename):
    """현재 프로젝트의 실험 노트북 코드만 실행해 메모리 변수를 가져옵니다."""
    notebook = json.loads((project_folder / "notebooks" / filename).read_text(encoding="utf-8"))
    namespace = {}
    with contextlib.redirect_stdout(io.StringIO()):
        for index, cell in enumerate(notebook["cells"]):
            if cell["cell_type"] == "code":
                exec(compile("".join(cell["source"]), f"{filename}:cell_{index}", "exec"), namespace)
    return namespace

type_example = run_notebook_source("05_chunk_types_and_token_budget.ipynb")
topic_example = run_notebook_source("04_topic_chunking.ipynb")
reader = type_example["reader"]
token_budget = type_example["token_budget"]
make_search_chunks = type_example["make_search_chunks"]
prepare_text = type_example["prepare_text"]

@lru_cache(maxsize=8192)
def token_count(text):
    """실제 토큰 길이를 중복 계산하지 않도록 메모리에서 잠시 기억합니다."""
    return len(type_example["tokenizer"](text, add_special_tokens=True, truncation=False, verbose=False)["input_ids"])

# 이전 분할 함수도 같은 토큰 계산을 사용하게 합니다.
type_example["token_count"] = token_count
print("PDF 전체 페이지:", len(reader.pages))
print("검색 조각 입력 한도:", token_budget, "토큰")

PDF 전체 페이지: 779
검색 조각 입력 한도: 128 토큰


### 2. 페이지·목차·본문 쪽수 목록 만들기

본문 숫자를 일괄 삭제하지 않습니다. 끝부분이 '장 이름 또는 장 번호 → 인쇄 쪽수 → 떨어진 글머리표'인 경우에만 꼬리말로 구분합니다. 인식하지 못한 경우에는 원문을 남기고 검토 대상으로 표시합니다.

목차는 점선과 참조 쪽수가 많은 페이지를 찾되, 이미 확인한 차량 안내도는 제외합니다. 뒤쪽 색인은 앞서 확인한 765쪽부터 제외합니다. 분류한 페이지의 원문과 그림 참조는 모두 목록에 남습니다.


꼬리말 뒤에 따로 추출된 단계 번호·각주·그림의 영문 표시는 페이지 목록에 보관하고 검토 표시를 남깁니다. 숫자 꼬리말 후보가 여러 개면 최종 쪽수 확인이 필요합니다.


In [2]:
# [프로젝트 추가 기록] 싼타페 목차 제목·인쇄 쪽수·장 번호를 이용해 전체 본문의 자동 주제 초안을 만듭니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
raw_pages = {number: page.extract_text() or "" for number, page in enumerate(reader.pages, start=1)}
row_pattern = re.compile(r"^([^\n]+?)\.{3,}\s*(\d+(?:\s*,\s*\d+)*)\s*$", re.MULTILINE)
known_overview_pages = {30, 31, 33, 34, 35, 36}
toc_pages = []
for number, text in raw_pages.items():
    rows = row_pattern.findall(text)
    line_count = max(1, sum(bool(line.strip()) for line in text.splitlines()))
    if (number < 765 and number not in known_overview_pages
            and len(rows) >= 2 and len(rows) / line_count > 0.4
            and len(reader.pages[number - 1].images) == 0):
        toc_pages.append(number)

# 이번 PDF에서 확인한 목차 목록과 달라지면 자동 처리를 멈추고 다시 살펴봅니다.
expected_toc_pages = [7, 29, 49, 50, 63, 147, 183, 184, 185, 186, 187, 379, 380, 461, 462, 463, 659, 693, 694]
assert toc_pages == expected_toc_pages

def normalize_title(text):
    """제목 비교에서는 공백 차이만 제거합니다. 내용이나 숫자는 바꾸지 않습니다."""
    return re.sub(r"\s+", "", text)

chapter_names = {}
for number in toc_pages:
    for chapter, name in re.findall(r"^([0-9])\.\s+([^\n]+)$", raw_pages[number], re.MULTILINE):
        chapter_names[normalize_title(name)] = int(chapter)

def split_footer(raw_text):
    """끝부분의 꼬리말 후보를 찾고, 따로 추출된 번호·각주도 원문 목록에 남깁니다."""
    lines = []
    position = 0
    for line in raw_text.splitlines(keepends=True):
        if line.strip():
            lines.append({"text": line.strip(), "start": position})
        position += len(line)

    candidates = []
    marker_pattern = r"[•-]|\*?\d+:|[A-Z]:|\d+"
    for index in range(len(lines) - 1):
        section = normalize_title(lines[index]["text"])
        chapter = chapter_names.get(section)
        if re.fullmatch(r"[0-9]", section):
            chapter = int(section)
        manual_text = lines[index + 1]["text"]
        if chapter is None or not manual_text.isdigit():
            continue
        manual_number = int(manual_text)
        if not 0 < manual_number <= len(reader.pages):
            continue
        tail = [line["text"] for line in lines[index + 2:]]
        if all(re.fullmatch(marker_pattern, marker) for marker in tail):
            candidates.append({
                "index": index, "chapter": chapter, "manual": manual_number,
                "named_section": section in chapter_names, "tail": tail,
            })
    if candidates:
        # 장 이름 후보를 우선합니다. 숫자 후보가 여러 개면 검토 표시를 남깁니다.
        candidates.sort(key=lambda c: (c["named_section"], c["manual"]), reverse=True)
        chosen = candidates[0]
        index = chosen["index"]
        ambiguous = len(candidates) > 1 and not chosen["named_section"]
        return (raw_text[:lines[index]["start"]], chosen["manual"], chosen["chapter"],
                True, ambiguous, chosen["tail"])
    return raw_text, None, None, False, False, []

page_inventory = {}
image_manifest = []
for number, raw_text in raw_pages.items():
    body, manual_number, chapter, footer_found, footer_review, detached_markers = split_footer(raw_text)
    # 이미지 키 목록은 픽셀을 모두 디코딩하지 않고 읽을 수 있습니다.
    parts = []
    for index, key in enumerate(reader.pages[number - 1].images.keys()):
        part = {
            "image_id": f"pdf_{number}_image_{index}",
            "pdf_page_number": number,
            "pdf_image_key": key,
            "name": None,
            "size": None,
            "local_path": None,
            "description": None,
        }
        parts.append(part)
        image_manifest.append(part)

    divider = re.fullmatch(r"([0-9])\.\s*([^\n]+)\s*", raw_text.strip())
    is_divider = bool(divider and normalize_title(divider.group(2)) in chapter_names)
    if not raw_text.strip() and not parts:
        role = "empty"
    elif number >= 765:
        role = "index"
    elif number in toc_pages:
        role = "toc"
    elif is_divider or (footer_found and not body.strip()):
        role = "divider_or_empty_body"
    elif number <= 6:
        role = "frontmatter"
    else:
        role = "body"
    page_inventory[number] = {
        "raw_text": raw_text, "body_text": body,
        "manual_page_number": manual_number, "chapter_id": chapter,
        "footer_found": footer_found, "footer_review_required": footer_review,
        "detached_markers": detached_markers, "role": role, "image_parts": parts,
    }

# 목차의 참조 쪽수와 본문 제목을 함께 비교할 색인을 만듭니다.
toc_entries = defaultdict(list)
for number in toc_pages:
    for title, pages in row_pattern.findall(raw_pages[number]):
        if "," in pages:
            continue
        toc_entries[(int(pages), normalize_title(title))].append({
            "title": re.sub(r"\s+", " ", title.strip()),
            "toc_pdf_page_number": number,
        })

print("페이지 분류:", dict(Counter(p["role"] for p in page_inventory.values())))
print("목차 쪽수:", toc_pages)
print("목차 제목·참조 쪽수 조합:", len(toc_entries))
print("꼬리말에서 인쇄 쪽수를 읽은 페이지:", sum(p["footer_found"] for p in page_inventory.values()))
print("그림이 있는 페이지:", sum(bool(p["image_parts"]) for p in page_inventory.values()))
print("전체 이미지 참조:", len(image_manifest))

페이지 분류: {'frontmatter': 4, 'empty': 2, 'toc': 19, 'body': 734, 'divider_or_empty_body': 5, 'index': 15}
목차 쪽수: [7, 29, 49, 50, 63, 147, 183, 184, 185, 186, 187, 379, 380, 461, 462, 463, 659, 693, 694]
목차 제목·참조 쪽수 조합: 516
꼬리말에서 인쇄 쪽수를 읽은 페이지: 739
그림이 있는 페이지: 509
전체 이미지 참조: 864


### 3. 앞서 확인한 기록을 재사용

04번의 여섯 주제와 05번의 다섯 원문 묶음을 재사용합니다. 이전에 확인한 차량 외부 I·II와 차대번호 예제도 실제 PDF에서 다시 읽습니다. 이 기록들이 차지하는 페이지는 자동 기록과 중복되지 않도록 제외합니다.

그림 설명은 사람이 앞서 확인한 설명만 사용합니다.

In [3]:
# [프로젝트 추가 기록] 싼타페 목차 제목·인쇄 쪽수·장 번호를 이용해 전체 본문의 자동 주제 초안을 만듭니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
verified_parents = []
for original in type_example["parent_records"]:
    record = {
        "content": original["content"], "raw_text": original["raw_text"],
        "metadata": dict(original["metadata"]), "image_parts": original["image_parts"],
    }
    record["metadata"]["verification_status"] = "sample_verified"
    verified_parents.append(record)

for index, original in enumerate(topic_example["topic_records"]):
    record = {
        "content": original["content"], "raw_text": original["raw_text"],
        "metadata": dict(original["metadata"]), "image_parts": original["image_parts"],
    }
    record["metadata"].update({
        "record_id": f"sample_topic_{index}",
        "verification_status": "sample_verified",
    })
    verified_parents.append(record)

for number, title, expected_count, manual_number in [
    (30, "차량 외부 I", 9, 25), (31, "차량 외부 II", 10, 26),
]:
    rows = row_pattern.findall(raw_pages[number])
    assert len(rows) == expected_count
    items = [
        {"number": index, "name": name.strip(),
         "target_manual_pages": [int(n) for n in pages.split(",")]}
        for index, (name, pages) in enumerate(rows, start=1)
    ]
    content = title + "\n이 그림은 실제 차량과 다를 수 있습니다.\n"
    content += "\n".join(
        f'{item["number"]}: {item["name"]} — 매뉴얼 ' +
        ", ".join(str(n) + "쪽" for n in item["target_manual_pages"])
        for item in items
    )
    verified_parents.append({
        "content": content, "raw_text": raw_pages[number],
        "metadata": {
            "record_id": f"overview_{number}", "title": title,
            "pdf_page_number": number, "manual_page_number": manual_number,
            "source_pages": [number], "content_type": "vehicle_overview_navigation",
            "navigation_items": items, "verification_status": "sample_verified",
        },
        "image_parts": type_example["image_parts"](number, {"I1.jpg": title + "의 주요 부품 위치를 번호로 표시한 안내도."}),
    })

vin_description = "그림의 화살표는 엔진과 차량 실내 사이 차체 패널에 있는 차대번호 타각 위치를 가리킨다."
verified_parents.append({
    "content": prepare_text(page_inventory[45]["body_text"]) + "\n[그림 설명] " + vin_description,
    "raw_text": raw_pages[45],
    "metadata": {
        "record_id": "vin_45", "title": "차대번호 (VIN)",
        "pdf_page_number": 45, "manual_page_number": 40, "source_pages": [45],
        "content_type": "figure_with_text", "verification_status": "sample_verified",
    },
    "image_parts": type_example["image_parts"](45, {"I1.jpg": vin_description}),
})
override_pages = {number for record in verified_parents for number in record["metadata"]["source_pages"]}
print("재사용한 확인 기록:", len(verified_parents))
print("자동 처리와 중복을 제외할 페이지:", sorted(override_pages))

재사용한 확인 기록: 14
자동 처리와 중복을 제외할 페이지: [30, 31, 33, 34, 35, 36, 44, 45, 52, 53, 54]


### 4. 목차 제목으로 자동 주제 초안 만들기

본문에 인쇄된 쪽수와 목차의 참조 쪽수가 같고, 본문 한 줄이 목차 제목과 같을 때 주제 시작으로 사용합니다. 제목을 찾지 못한 글도 버리지 않고 임시 주제로 보관합니다.

이 규칙은 계층형 제목이나 줄이 바뀐 제목을 모두 이해하지는 못합니다. 긴 주제·표 후보·제목만 있는 짧은 주제·그림 연결은 검토 표시를 남깁니다. 그림은 같은 페이지에 있다는 관계만 자동으로 기록하며, 어떤 문장과 정확히 대응하는지는 주장하지 않습니다.


엔진룸 안내도처럼 긴 점선이 항목 이름에 붙어 있으면 점선만 참조 쪽수 표현으로 정리합니다. 항목과 참조 쪽수의 같은 줄 관계는 유지하되, 그림 번호 대응은 자동 승인하지 않습니다.


In [4]:
# [프로젝트 추가 기록] 싼타페 목차 제목·인쇄 쪽수·장 번호를 이용해 전체 본문의 자동 주제 초안을 만듭니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
auto_topics = []
active = None

def add_segment(topic, number, start, end):
    """본문의 원문 구간과 글자 위치를 보존합니다. 숫자 위치는 본문 문자열 기준입니다."""
    if end > start:
        topic["segments"].append({
            "pdf_page_number": number, "start": start, "end": end,
            "raw_text": page_inventory[number]["body_text"][start:end],
        })

def finish_active(topic):
    """완성된 임시 주제를 목록에 남깁니다."""
    if topic is not None and topic["segments"]:
        auto_topics.append(topic)

for number, page in page_inventory.items():
    if number in override_pages or page["role"] not in {"body", "frontmatter"}:
        finish_active(active)
        active = None
        continue
    body = page["body_text"]
    if not body.strip():
        continue
    if active is not None and page["chapter_id"] != active["chapter_id"]:
        finish_active(active)
        active = None

    matches = []
    offset = 0
    for line in body.splitlines(keepends=True):
        key = (page["manual_page_number"], normalize_title(line.strip()))
        if key in toc_entries:
            matches.append((offset, toc_entries[key][0]["title"]))
        offset += len(line)
    first_start = matches[0][0] if matches else len(body)

    # 새 페이지 제목 앞의 이어지는 설명은 같은 장의 앞 주제에 연결합니다.
    if first_start > 0:
        if active is None:
            active = {
                "title": f"PDF {number}쪽 제목 미확인 구간",
                "chapter_id": page["chapter_id"], "segments": [],
                "boundary_source": "unmatched_prefix",
            }
        add_segment(active, number, 0, first_start)
    for index, (start, title) in enumerate(matches):
        finish_active(active)
        active = {
            "title": title, "chapter_id": page["chapter_id"], "segments": [],
            "boundary_source": "toc_title_and_printed_page",
        }
        end = matches[index + 1][0] if index + 1 < len(matches) else len(body)
        add_segment(active, number, start, end)
finish_active(active)

auto_parents = []
for index, topic in enumerate(auto_topics):
    segments = topic["segments"]
    pages = list(dict.fromkeys(segment["pdf_page_number"] for segment in segments))
    raw_text = "\n".join(segment["raw_text"] for segment in segments)
    # 검색을 방해하는 긴 점선만 정리하고, 같은 줄의 항목 이름과 참조 숫자는 유지합니다.
    normalized_rows = row_pattern.sub(
        lambda match: match.group(1).strip() + " — 매뉴얼 " + match.group(2) + "쪽", raw_text
    )
    content = prepare_text(normalized_rows)
    flags = []
    if topic["boundary_source"] == "unmatched_prefix":
        flags.append("heading_not_confirmed")
    if any(not page_inventory[n]["footer_found"] for n in pages):
        flags.append("footer_not_confirmed")
    if any(page_inventory[n]["footer_review_required"] for n in pages):
        flags.append("ambiguous_footer_candidate_needs_review")
    if any(re.fullmatch(r"\*?\d+:|[A-Z]:|\d+", marker)
           for n in pages for marker in page_inventory[n]["detached_markers"]):
        flags.append("detached_number_or_letter_labels_needs_review")
    if len(pages) > 4:
        flags.append("long_topic_boundary_needs_review")
    if len(content) < 40:
        flags.append("thin_topic_needs_review")
    if row_pattern.search(raw_text) or re.search(r"종류\s+용량\s+추천 사양", raw_text):
        flags.append("table_or_navigation_structure_needs_review")
    # 실제 출력에서 확인한 표 제목과 깨진 방향 기호는 별도 검토 대상입니다.
    if topic["title"] == "주행 모드별 패들 쉬프트의 기능" and "table_or_navigation_structure_needs_review" not in flags:
        flags.append("table_or_navigation_structure_needs_review")
    if any(symbol in content for symbol in ["î", "ï"]):
        flags.append("font_symbol_needs_review")
    if any(re.search(r"(?:҃Ҋ|઱੄|ӝ)\s*$", segment["raw_text"]) for segment in segments):
        flags.append("warning_boundary_needs_review")

    images = [part for n in pages for part in page_inventory[n]["image_parts"]]
    if images:
        flags.append("page_level_image_linkage_needs_review")
    auto_parents.append({
        "content": content,
        "raw_text": raw_text,
        "metadata": {
            "record_id": f"auto_topic_{index}",
            "title": topic["title"],
            "pdf_page_number": pages[0],
            "manual_page_number": page_inventory[pages[0]]["manual_page_number"],
            "source_pages": pages,
            "chapter_id": topic["chapter_id"],
            "content_type": "figure_with_text" if images else "instruction_text",
            "boundary_source": topic["boundary_source"],
            "source_spans": [
                {k: segment[k] for k in ["pdf_page_number", "start", "end"]}
                for segment in segments
            ],
            "verification_status": "auto_draft_needs_review",
            "review_flags": flags,
        },
        "image_parts": images,
    })

full_parent_records = verified_parents + auto_parents
print("전체 원문 묶음:", len(full_parent_records))
print("확인 예제:", len(verified_parents), "| 자동 초안:", len(auto_parents))
print("목차와 인쇄 쪽수로 시작한 초안:", sum(p["metadata"]["boundary_source"] == "toc_title_and_printed_page" for p in auto_parents))
print("검토 표시:", dict(Counter(flag for p in auto_parents for flag in p["metadata"]["review_flags"])))

전체 원문 묶음: 524
확인 예제: 14 | 자동 초안: 510
목차와 인쇄 쪽수로 시작한 초안: 507
검토 표시: {'heading_not_confirmed': 3, 'footer_not_confirmed': 2, 'page_level_image_linkage_needs_review': 374, 'thin_topic_needs_review': 49, 'detached_number_or_letter_labels_needs_review': 175, 'table_or_navigation_structure_needs_review': 3, 'long_topic_boundary_needs_review': 34, 'ambiguous_footer_candidate_needs_review': 64, 'font_symbol_needs_review': 8}


### 5. 모델 입력 길이에 맞는 검색 조각 미리보기

원문 전체는 부모 기록에 남기고 짧은 검색 조각을 만듭니다. 원문 묶음이 자동 초안이면 검색 조각에도 같은 검토 상태를 전달합니다. 분할하지 못한 기록도 버리지 않고 오류 목록에 남깁니다.

여기서는 토큰 길이만 확인하며 전체 임베딩 계산이나 답변 생성은 하지 않습니다.

In [5]:
# [프로젝트 추가 기록] 싼타페 목차 제목·인쇄 쪽수·장 번호를 이용해 전체 본문의 자동 주제 초안을 만듭니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
full_search_chunks = []
chunking_errors = []
for parent in full_parent_records:
    try:
        children = make_search_chunks(parent, token_budget)
        for child in children:
            child["metadata"]["verification_status"] = parent["metadata"]["verification_status"]
            child["metadata"]["review_flags"] = parent["metadata"].get("review_flags", [])
        full_search_chunks.extend(children)
    except ValueError as error:
        parent["metadata"].setdefault("review_flags", []).append("chunking_error_needs_review")
        chunking_errors.append({
            "record_id": parent["metadata"]["record_id"],
            "title": parent["metadata"]["title"], "error": str(error),
        })

assert all(child["metadata"]["token_count"] <= token_budget for child in full_search_chunks)
print("검색 조각 수:", len(full_search_chunks))
print("최대 검색 조각 길이:", max(c["metadata"]["token_count"] for c in full_search_chunks), "토큰")
print("분할 오류 기록:", len(chunking_errors))
for error in chunking_errors:
    print("분할 오류:", error)
print("아직 전체 임베딩이나 DB 저장은 실행하지 않았습니다.")

검색 조각 수: 2209
최대 검색 조각 길이: 128 토큰
분할 오류 기록: 0
아직 전체 임베딩이나 DB 저장은 실행하지 않았습니다.


### 6. 원문 구간 누락과 이미지 참조 확인

자동 처리 대상 본문의 모든 구간이 한 번씩 사용됐는지 확인합니다. 목차·색인·확인 예제의 페이지는 그 검사의 대상과 구분됩니다. 제외한 페이지도 페이지 목록에서 원문과 그림을 확인할 수 있습니다.

이미지 키는 나중에 실제 그림을 PDF에서 읽을 때 사용하는 주소입니다. 이름이나 크기가 None인 참조는 아직 이미지를 디코딩하지 않았다는 뜻이며, 이미지가 사라졌다는 뜻이 아닙니다.

In [6]:
# [프로젝트 추가 기록] 싼타페 목차 제목·인쇄 쪽수·장 번호를 이용해 전체 본문의 자동 주제 초안을 만듭니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
coverage = defaultdict(list)
for parent in auto_parents:
    for span in parent["metadata"]["source_spans"]:
        coverage[span["pdf_page_number"]].append((span["start"], span["end"]))
for number, page in page_inventory.items():
    if number in override_pages or page["role"] not in {"body", "frontmatter"} or not page["body_text"].strip():
        continue
    ranges = sorted(coverage[number])
    assert ranges and ranges[0][0] == 0
    assert ranges[-1][1] == len(page["body_text"])
    assert all(left[1] == right[0] for left, right in zip(ranges, ranges[1:]))

ids = [parent["metadata"]["record_id"] for parent in full_parent_records]
assert len(ids) == len(set(ids))
assert len(image_manifest) == 864
assert sum(bool(p["image_parts"]) for p in page_inventory.values()) == 509
assert len(page_inventory) == 779

# 각 원문의 자식 조각 구간도 연속해서 부모 검색 글을 덮는지 확인합니다.
chunk_groups = defaultdict(list)
for child in full_search_chunks:
    chunk_groups[child["metadata"]["parent_record_id"]].append(child)
failed_ids = {item["record_id"] for item in chunking_errors}
for parent in full_parent_records:
    parent_id = parent["metadata"]["record_id"]
    if parent_id in failed_ids:
        continue
    cursor = 0
    for child in chunk_groups[parent_id]:
        assert child["metadata"]["parent_content_start"] == cursor
        cursor = child["metadata"]["parent_content_end"]
    assert cursor == len(parent["content"])

print("자동 본문 구간 누락·중복:", "없음")
print("검사한 자동 처리 페이지:", len(coverage))
print("원문 ID 중복:", "없음")
print("전체 페이지:", len(page_inventory))
print("전체 이미지 참조:", len(image_manifest))
print("\n살펴볼 주제 예시:")
for title in ["패들 쉬프트", "추천 오일 및 용량", "시동 버튼"]:
    examples = [p for p in full_parent_records if title in p["metadata"]["title"]]
    for parent in examples[:2]:
        print(parent["metadata"]["title"], "| PDF:", parent["metadata"]["source_pages"],
              "| 상태:", parent["metadata"]["verification_status"],
              "| 검토:", parent["metadata"].get("review_flags", []))
        print("글 앞부분:", parent["content"][:180])
print("\n메모리 자료:", "page_inventory / full_parent_records / full_search_chunks / image_manifest")
print("파일·Supabase 저장:", "실행하지 않음")

자동 본문 구간 누락·중복: 없음
검사한 자동 처리 페이지: 727
원문 ID 중복: 없음
전체 페이지: 779
전체 이미지 참조: 864

살펴볼 주제 예시:
패들 쉬프트 (수동 변속 모드) | PDF: [397, 398] | 상태: auto_draft_needs_review | 검토: ['detached_number_or_letter_labels_needs_review', 'font_symbol_needs_review', 'page_level_image_linkage_needs_review']
글 앞부분: 패들 쉬프트 (수동 변속 모드) 패들 쉬프트는 스티어링 휠에서 손을 떼지 않고 변속할 수 있는 변속 장치로 다음과 같이 작동합니 다. 선택된 드라이브 모드에 따라 패들 쉬프트의 기능이 달라집니다. 수동 변속 모드는 SPORT 모드 에서만 작동합니다. 자동 변속 모드에서 î 또는 ï 레버를 한 번 당기면 한 단 올라가거나 
주행 모드별 패들 쉬프트의 기능 | PDF: [402] | 상태: auto_draft_needs_review | 검토: ['table_or_navigation_structure_needs_review']
글 앞부분: 주행 모드별 패들 쉬프트의 기능 [참고] 선택된 드라이브 모드에 따라 패들 쉬프트의 기능이 달라집니다. 드라이브 모드 패들 쉬프트 레버 조작 패들 쉬프트 레버 기능 ECO + 회생 제동 단계 감소 - 회생 제동 단계 증가 SPORT + 수동 변속 + - 수동 변속 - ECO 모드에서 시동을 껐다가 다시 걸면 ECO 모드가
추천 오일 및 용량 | PDF: [43] | 상태: auto_draft_needs_review | 검토: ['detached_number_or_letter_labels_needs_review', 'table_or_navigation_structure_needs_review']
글 앞부분: 추천 오일 및 용량 종류 용량 추천 사양 연료 67 ℓ 무연 휘발유 엔진 오일 *1 4.8 ℓ SAE 0W-20 API 

이 단계는 전체 문서를 검색할 자료로 준비하는 초안입니다. 원문 구간 보존과 토큰 한도 확인을 통과해도, 자동 제목 경계와 그림 관계가 모두 정확하다는 뜻은 아닙니다.

다음에는 상세 설명이 나오는 주제와 표 후보를 직접 살펴보고, 검토 상태를 정리한 뒤 전체 임베딩으로 넘어갑니다. 08번의 좁은 답변 규칙을 이 자동 자료 전체에 바로 적용하지 않습니다.

## 이번 실행 결과와 다음 작업

- 전체 PDF 779쪽을 분류하고 원문 묶음 524개를 만들었습니다. 이 중 14개는 앞서 확인한 예제이며, 510개는 원본 대조가 필요한 자동 초안입니다.
- 검색 조각은 2,209개이며 모두 현재 임베딩 모델의 입력 한도인 128토큰 이내입니다. 분할 오류는 0개입니다.
- 자동 처리 대상 727쪽의 본문 구간에 누락이나 중복이 없는지 확인했습니다. 이 검사는 글자 구간의 보존 여부를 확인하며, 주제 구분의 정확성을 보장하지는 않습니다.
- 이미지 참조 864개를 목록에 남겼습니다. 자동 초안의 이미지 연결은 페이지 기준이므로, 각 그림이 어떤 설명에 해당하는지 추가로 확인해야 합니다.
- 이미지 연결 검토 374개, 꼬리말 후보 검토 64개, 짧은 주제 검토 49개, 깨진 글꼴 기호 검토 8개가 표시되었습니다. 한 기록에 여러 표시가 있을 수 있습니다.
- 데이터 파일 저장, 전체 임베딩 계산, SQL/Supabase 저장은 아직 진행하지 않았습니다.

### 다음 순서

1. PDF 397~398쪽의 패들 쉬프트 설명에서 깨진 방향 기호를 원본과 대조합니다.
2. PDF 402쪽의 주행 모드별 표와 43쪽의 추천 오일 및 용량 표를 원본과 대조합니다.
3. 제목만 있는 짧은 묶음과 이미지·설명 연결을 정리한 뒤, 전체 임베딩 단계로 넘어갑니다.
